In [ ]:
import os

# 1. Clone repository
if not os.path.exists("/kaggle/working/voicebox"):
    !git clone https://github.com/jamiepine/voicebox.git

%cd /kaggle/working/voicebox

# 2. Install audio codecs and phonemizer required by TTS engines
!apt-get update -qq && apt-get install -y -qq ffmpeg sox libsox-fmt-all espeak-ng

# 3. Install Bun runtime & compile the Web UI frontend
print("--- Building Frontend SPA ---")
!curl -fsSL https://bun.sh/install | bash
!export PATH="$HOME/.bun/bin:$PATH" && bun install
!export PATH="$HOME/.bun/bin:$PATH" && cd web && bunx --bun vite build

# 4. Create native frontend symlink expected by backend/app.py
frontend_target = "/kaggle/working/voicebox/frontend"
if os.path.exists(frontend_target) or os.path.islink(frontend_target):
    !rm -rf {frontend_target}
!ln -s /kaggle/working/voicebox/web/dist {frontend_target}
print("Frontend setup complete!")

In [ ]:
# 1. Install uv for fast virtual environment management
!pip install -q uv

# 2. Create Python 3.11 virtual environment
print("--- Setting up Python 3.11 Environment ---")
!uv venv /kaggle/working/venv --python 3.11

# 3. Install requirements inside the venv
print("--- Installing Backend Requirements ---")
!uv pip install --python /kaggle/working/venv/bin/python -r /kaggle/working/voicebox/backend/requirements.txt

In [ ]:
import os
import subprocess
import time
import urllib.request
import re

# 1. Stop any lingering instances
!fuser -k 17493/tcp >/dev/null 2>&1 || true
!killall cloudflared >/dev/null 2>&1 || true

# 2. Configure Environment Variables
env = os.environ.copy()
env["VOICEBOX_MODELS_DIR"] = "/kaggle/working/models_cache"
env["VOICEBOX_CORS_ORIGINS"] = "*"
env["PYTHONPATH"] = "/kaggle/working/voicebox"
os.makedirs("/kaggle/working/models_cache", exist_ok=True)

# 3. Start Uvicorn Backend Server
print("--- Starting Voicebox Backend Server ---")
log_path = "/kaggle/working/backend.log"
log_file = open(log_path, "w")

server_proc = subprocess.Popen(
    ["/kaggle/working/venv/bin/python", "-u", "-m", "uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "17493"],
    cwd="/kaggle/working/voicebox",
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env
)

# 4. Verify local server is serving HTML
print("Verifying Web UI response...", end="")
server_ready = False
for _ in range(15):
    time.sleep(2)
    try:
        with urllib.request.urlopen("http://127.0.0.1:17493/", timeout=2) as r:
            body = r.read().decode("utf-8", errors="ignore")
            if "<html" in body.lower() or "<!doctype html" in body.lower():
                server_ready = True
                print(" ✅ Success! Web UI is live.")
                break
    except Exception:
        pass

if not server_ready:
    print("\n❌ Server failed to serve HTML. Here is backend.log:")
    with open(log_path, "r") as f:
        print(f.read())
    raise RuntimeError("Voicebox backend failed verification.")

# 5. Download and launch Cloudflare Tunnel
if not os.path.exists("/kaggle/working/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
    !chmod +x /kaggle/working/cloudflared

print("--- Launching Public HTTPS Tunnel ---")
tunnel_proc = subprocess.Popen(
    ["/kaggle/working/cloudflared", "tunnel", "--url", "http://127.0.0.1:17493"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        print(f"\n=======================================================")
        print(f"🚀 OPEN VOICEBOX WEB STUDIO: {match.group(0)}")
        print(f"=======================================================\n")
        break

# Keep cell alive so server and tunnel stay running
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    server_proc.terminate()
    tunnel_proc.terminate()